# Workshop 4 Live Demo: Connect Everything

## From Source Code to Result

In Workshop 3, we built:

```text
source code → tokens → AST
```

In this live demo, we connect the whole pipeline:

```text
source code → tokens → parser → AST → evaluator → result
```

## Demo Plan

1. Define tokens and AST nodes.
2. Use a tiny lexer.
3. Use a tiny parser.
4. Implement `evaluate()`.
5. Connect everything inside `run()`.
6. Show inspect mode.
7. Preview variables and environments.

In [25]:
from dataclasses import dataclass
from enum import Enum, auto
from typing import Any

# Part 1 — Tokens and AST Nodes

These are the basic data structures from Workshop 3.

In [26]:
class TokenType(Enum):
    NUMBER = auto()
    PLUS = auto()
    STAR = auto()
    EOF = auto()


@dataclass(frozen=True)
class Token:
    type: TokenType
    value: Any = None

    def __repr__(self):
        if self.value is None:
            return self.type.name
        return f"{self.type.name}({self.value})"

In [27]:
@dataclass(frozen=True)
class Number:
    value: int


@dataclass(frozen=True)
class BinaryOp:
    op: str
    left: Any
    right: Any

# Part 2 — Lexer

The lexer turns source code text into tokens.

Lecture prompt:

> What should `tokenize("12 + 3")` produce?

In [28]:
def tokenize(source: str) -> list[Token]:
    tokens = []
    i = 0

    while i < len(source):
        char = source[i]

        if char.isspace():
            i += 1
            continue

        if char.isdigit():
            start = i
            while i < len(source) and source[i].isdigit():
                i += 1
            tokens.append(Token(TokenType.NUMBER, int(source[start:i])))
            continue

        if char == "+":
            tokens.append(Token(TokenType.PLUS))
            i += 1
            continue

        if char == "*":
            tokens.append(Token(TokenType.STAR))
            i += 1
            continue

        raise SyntaxError(f"Unexpected character: {char!r}")

    tokens.append(Token(TokenType.EOF))
    return tokens

In [29]:
tokens = tokenize("1 + 2 * 3")
tokens

[NUMBER(1), PLUS, NUMBER(2), STAR, NUMBER(3), EOF]

# Part 3 — Parser

Grammar:

```text
expression → term (+ term)*
term       → factor (* factor)*
factor     → NUMBER
```

Because `*` lives deeper in the grammar, it binds more tightly.

In [7]:
class Parser:
    def __init__(self, tokens: list[Token]):
        self.tokens = tokens
        self.current = 0

    def peek(self) -> Token:
        return self.tokens[self.current]

    def advance(self) -> Token:
        token = self.peek()
        self.current += 1
        return token

    def match(self, token_type: TokenType) -> bool:
        if self.peek().type == token_type:
            self.advance()
            return True
        return False

    def consume(self, token_type: TokenType, message: str) -> Token:
        if self.peek().type == token_type:
            return self.advance()
        raise SyntaxError(message)

    def parse(self):
        expr = self.parse_expression()
        self.consume(TokenType.EOF, "Expected end of expression")
        return expr

    def parse_expression(self):
        left = self.parse_term()

        while self.match(TokenType.PLUS):
            right = self.parse_term()
            left = BinaryOp("+", left, right)

        return left

    def parse_term(self):
        left = self.parse_factor()

        while self.match(TokenType.STAR):
            right = self.parse_factor()
            left = BinaryOp("*", left, right)

        return left

    def parse_factor(self):
        token = self.peek()

        if token.type == TokenType.NUMBER:
            self.advance()
            return Number(token.value)

        raise SyntaxError(f"Expected number, got {token}")

In [30]:
ast = Parser(tokenize("1 + 2 * 3")).parse()
ast

BinaryOp(op='+', left=Number(value=1), right=BinaryOp(op='*', left=Number(value=2), right=Number(value=3)))

## Pretty Print the AST

This helper makes the tree easier to discuss during lecture.

In [31]:
def pretty_ast(node, indent: str = "") -> str:
    if isinstance(node, Number):
        return f"{indent}Number({node.value})"

    if isinstance(node, BinaryOp):
        left = pretty_ast(node.left, indent + "  ")
        right = pretty_ast(node.right, indent + "  ")

        return (
            f"{indent}BinaryOp(\n"
            f"{indent}  op={node.op!r},\n"
            f"{indent}  left=\n{left},\n"
            f"{indent}  right=\n{right}\n"
            f"{indent})"
        )

    raise TypeError(f"Unknown AST node: {node}")

In [10]:
print(pretty_ast(ast))

BinaryOp(
  op='+',
  left=
  Number(1),
  right=
  BinaryOp(
    op='*',
    left=
    Number(2),
    right=
    Number(3)
  )
)


# Part 4 — Live Coding: Evaluator

This is the main live-coding moment.

In [37]:
# Starter version for live coding

def evaluate_live(node):
    if isinstance(node, Number):
        return node.value
    

    # TODO 2:
    if isinstance(node, BinaryOp):
        
    # If this is a BinaryOp:
    #   evaluate the left child 
        left = evaluate_live(node.left)
    #   evaluate the right child
        right = evaluate_live(node.right)
        if node.op == "+":
            return left + right
        if node.op == "*":
            return left * right
   
    #   combine based on the operator

    raise NotImplementedError("Live-code this function")

In [38]:
evaluate_live(ast)

7

# Part 5 — Connect Everything

Now we combine all stages into one function.

Lecture prompt:

> Which function runs first?
> Which function runs last?

In [49]:
def run(source: str):
    tokens = tokenize(source)
    ast = Parser(tokens).parse()
    result = evaluate_live(ast)
    return result

In [50]:
run("1 + 2 * 3")

7

In [52]:
while True:
    program = input()
    print(run(program))

 1+2


3


KeyboardInterrupt: Interrupted by user

In [ ]:
examples = [
    "1 + 2",
    "2 * 3",
    "10 + 20 * 3",
    "1 + 2 + 3",
    "2 * 3 * 4",
]

for source in examples:
    print(source, "=>", run(source))

# Part 6 — Inspect Mode

Sometimes we don't just want the result.

We want to see every stage.

In [44]:
def inspect(source: str):
    print("Source:")
    print(source)

    print("\nTokens:")
    tokens = tokenize(source)
    print(tokens)

    print("\nAST:")
    ast = Parser(tokens).parse()
    print(pretty_ast(ast))

    print("\nResult:")
    print(evaluate(ast))

In [45]:
inspect("1 + 2 * 3")

Source:
1 + 2 * 3

Tokens:
[NUMBER(1), PLUS, NUMBER(2), STAR, NUMBER(3), EOF]

AST:
BinaryOp(
  op='+',
  left=
  Number(1),
  right=
  BinaryOp(
    op='*',
    left=
    Number(2),
    right=
    Number(3)
  )
)

Result:


TypeError: evaluate() missing 1 required positional argument: 'env'

# Part 7 — Mini REPL-Style Demo

This simulates a few REPL commands without using `input()`.

In [43]:
commands = [
    "1 + 2",
    "1 + 2 * 3",
    "10 * 2 + 5",
]

for command in commands:
    print(">>>", command)
    print(run(command))
    print()

>>> 1 + 2
3

>>> 1 + 2 * 3
7

>>> 10 * 2 + 5
25



# Part 8 — Preview: Why Variables Need an Environment

Right now, every run is independent.

Next, we want:

```text
>>> x = 5
>>> x + 2
7
```

That requires an environment:

```python
env = {"x": 5}
```

In [58]:
@dataclass
class Variable:
    name: str

In [59]:
class Environment:
    def __init__(self):
        self.values = {}

    def define(self, name, value):
        self.values[name] = value

    def get(self, name):
        if name not in self.values:
            raise NameError(f"{name} is not defined")
        return self.values[name]

In [62]:
env = Environment()

# TODO: define variables
env.define('x',5)
env.define('y',10)

print(env.values)

{'x': 5, 'y': 10}


In [71]:
def evaluate(node, env):
    if isinstance(node, Number):
        return node.value

    if isinstance(node, BinaryOp):
        left = evaluate(node.left, env)
        right = evaluate(node.right, env)

        if node.op == "+":
            return left + right

        if node.op == "-":
            return left - right

        if node.op == "*":
            return left * right

        if node.op == "/":
            return left / right


    if isinstance(node, Variable):
        return env.get(node.name)


In [72]:
evaluate(Variable("x"), env)

5

In [73]:
ast = BinaryOp(
    "*",
    BinaryOp(
        "+",
        Variable("x"),
        Number(3),
    ),
    Variable("y"),
)

evaluate(ast, env)

80

In [ ]:
y-> 10  * (3 + x-> 5)